# CS 521 HW3, Problem 1: PGD Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

The code base is the starter notebook of Problem 3 (same data, model and attacks).
Here the models are trained **from scratch** (`make_model()`), not from the base model of Problem 3:

* a standard model (cross-entropy on clean images), and
* an $\ell_\infty$ adversarially trained model (cross-entropy on 10-step PGD examples, $\epsilon = 8/255$),

each for 5 epochs on the 10,000 training images, and evaluated on the 1,000 test images with
PGD and FGSM (one PGD step) for different $\epsilon$.

# Set up for dataset and model

In [1]:
# (The files are only downloaded if they are not here yet.)
![ -f cifar10_train_10k.pt ] || gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
![ -f cifar10_test_1k.pt ] || gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: mps


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)

# Attacks

`pgd_linf` and `pgd_l2` are the PGD attacks from HW1 (same code as in Problem 3).
FGSM is one step of untargeted PGD with step size $\epsilon$ and no random start: $x + \epsilon \cdot \mathrm{sign}(\nabla_x \mathcal{L})$.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        # Random direction, random radius in [0, eps] (same scheme as the l1 attack).
        delta = torch.randn_like(x)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(eps))  # scale back only if ||delta||_2 > eps
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluation

Accuracy (in %) on the 1,000 test images under untargeted $\ell_\infty$ attacks:
FGSM (1 step, no random start) and PGD (20 steps, step size $2.5\epsilon/20$, random start).
$\epsilon$ is given in units of 1/255; $\epsilon = 0$ is the standard (clean) accuracy.

In [7]:
EVAL_EPS = [0, 1, 2, 4, 8, 12, 16]  # in units of 1/255


def linf_acc(model, eps, steps, alpha, rand_init, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts for each model
    correct = 0
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        x_adv = x if eps == 0 else pgd_linf(model, x, y, eps, alpha, steps, rand_init)
        with torch.no_grad():
            correct += (model(x_adv).argmax(1) == y).sum().item()
    return 100.0 * correct / len(y_test)


def fgsm_acc(model, eps):
    return linf_acc(model, eps, steps=1, alpha=eps, rand_init=False)


def pgd_acc(model, eps, steps=20):
    return linf_acc(model, eps, steps, alpha=2.5 * eps / steps, rand_init=True)


def eps_sweep(models):
    print(f"{'model':<18}{'attack':<7}" + "".join(f"{e:>3}/255" for e in EVAL_EPS))
    for name, model in models.items():
        for attack_name, fn in [("FGSM", fgsm_acc), ("PGD", pgd_acc)]:
            accs = [fn(model, e / 255) for e in EVAL_EPS]
            print(f"{name:<18}{attack_name:<7}" + "".join(f"{a:7.1f}" for a in accs))

# Training

`train(model, loss_fn)` is the training loop of Problem 3 (`adv_train`) without the RAMP part.
The models are trained from scratch, so the learning rate is 0.1 (instead of 0.01 for fine-tuning).

In [8]:
def train(model, loss_fn, epochs=5, lr=0.1, batch_size=128, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model


def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


def linf_adv_loss(model, x, y, eps=EPS["linf"], steps=10):
    # Adversarial training: replace the batch by its PGD adversarial examples (made in eval mode).
    model.eval()
    x_adv = pgd_linf(model, x, y, eps, alpha=2.5 * eps / steps, steps=steps)
    model.train()
    return F.cross_entropy(model(x_adv), y)

## Part (a): standard model and $\ell_\infty$ adversarially trained model ($\epsilon = 8/255$)

In [9]:
torch.manual_seed(0)
model_std = train(make_model(), standard_loss)

epoch 1: loss 1.922 (9 s)


epoch 2: loss 1.653 (17 s)


epoch 3: loss 1.501 (26 s)


epoch 4: loss 1.344 (35 s)


epoch 5: loss 1.219 (43 s)


In [10]:
torch.manual_seed(0)
model_adv = train(make_model(), linf_adv_loss)

epoch 1: loss 2.203 (63 s)


epoch 2: loss 2.103 (126 s)


epoch 3: loss 2.036 (196 s)


epoch 4: loss 1.990 (261 s)


epoch 5: loss 1.959 (322 s)


In [11]:
eps_sweep({"standard": model_std, "linf AT (8/255)": model_adv})

model             attack   0/255  1/255  2/255  4/255  8/255 12/255 16/255


standard          FGSM      54.4   43.8   33.4   19.4    5.9    3.7    2.8


standard          PGD       54.4   43.3   31.6   14.9    2.4    0.5    0.1


linf AT (8/255)   FGSM      39.1   37.4   36.3   32.1   25.3   20.5   16.6


linf AT (8/255)   PGD       39.1   37.4   36.3   32.4   25.7   19.4   14.6


## Part (b): different training $\epsilon$

Two more $\ell_\infty$ adversarially trained models with a smaller and a larger training $\epsilon$,
evaluated with the same attacks.

In [12]:
torch.manual_seed(0)
model_adv4 = train(make_model(), linf_adv_loss, eps=4 / 255)
torch.manual_seed(0)
model_adv16 = train(make_model(), linf_adv_loss, eps=16 / 255)
eps_sweep({"linf AT (4/255)": model_adv4, "linf AT (16/255)": model_adv16})

epoch 1: loss 2.082 (80 s)


epoch 2: loss 1.952 (150 s)


epoch 3: loss 1.858 (211 s)


epoch 4: loss 1.796 (273 s)


epoch 5: loss 1.751 (335 s)


...

epoch 5: loss 2.196 (305 s)
model             attack   0/255  1/255  2/255  4/255  8/255 12/255 16/255


linf AT (4/255)   FGSM      45.2   42.3   39.4   34.4   25.6   16.8   12.1


linf AT (4/255)   PGD       45.2   42.4   39.6   34.3   23.8   15.0    8.4


linf AT (16/255)  FGSM      29.2   29.1   28.1   26.9   24.7   23.2   21.6


linf AT (16/255)  PGD       29.2   29.1   28.0   26.9   24.5   23.4   21.7


## Observations

All models are trained from scratch (`make_model()`) for 5 epochs on the 10,000 training images, and evaluated on the 1,000 test images.
The accuracies are in %. PGD uses 20 steps with a random start, and FGSM is one step without a random start.

| model | attack | 0 | 1/255 | 2/255 | 4/255 | 8/255 | 12/255 | 16/255 |
|---|---|---|---|---|---|---|---|---|
| standard | FGSM | 54.4 | 43.8 | 33.4 | 19.4 | 5.9 | 3.7 | 2.8 |
| standard | PGD | 54.4 | 43.3 | 31.6 | 14.9 | 2.4 | 0.5 | 0.1 |
| $\ell_\infty$ AT, 8/255 | FGSM | 39.1 | 37.4 | 36.3 | 32.1 | 25.3 | 20.5 | 16.6 |
| $\ell_\infty$ AT, 8/255 | PGD | 39.1 | 37.4 | 36.3 | 32.4 | 25.7 | 19.4 | 14.6 |
| $\ell_\infty$ AT, 4/255 | FGSM | 45.2 | 42.3 | 39.4 | 34.4 | 25.6 | 16.8 | 12.1 |
| $\ell_\infty$ AT, 4/255 | PGD | 45.2 | 42.4 | 39.6 | 34.3 | 23.8 | 15.0 | 8.4 |
| $\ell_\infty$ AT, 16/255 | FGSM | 29.2 | 29.1 | 28.1 | 26.9 | 24.7 | 23.2 | 21.6 |
| $\ell_\infty$ AT, 16/255 | PGD | 29.2 | 29.1 | 28.0 | 26.9 | 24.5 | 23.4 | 21.7 |

Training time (Apple M2 GPU): 43 s for the standard model, and about 320 s for each adversarially trained model.
Each training step of adversarial training also runs a 10-step PGD attack (10 more forward and backward passes), so it is about 7.5× slower.

### Part (a): standard vs adversarially trained model

* **Standard accuracy.** The standard model has 54.4% clean accuracy, and the $\ell_\infty$ AT model ($\epsilon = 8/255$) has 39.1%. Adversarial training costs 15 points of clean accuracy.
  The training loss shows the reason. The AT model fits the adversarial examples much more slowly (final loss 1.96, against 1.22 for the standard model), because it must learn a function that is constant in a ball around each image.
  Both accuracies are low because the training is short: 5 epochs on 10,000 images.
* **Robust accuracy.** Small perturbations are enough to break the standard model. PGD with $\epsilon = 2/255$ takes it from 54.4% to 31.6%, $\epsilon = 4/255$ to 14.9%, and $\epsilon = 8/255$ to 2.4%.
  The AT model loses accuracy slowly: 37.4% at 1/255, 32.4% at 4/255, and 25.7% at 8/255, the $\epsilon$ it was trained for. That is about 10× the standard model at 8/255.
* **Crossover.** The AT model is better than the standard model for every $\epsilon \ge 2/255$. For $\epsilon \le 1/255$, the standard model is still better, because its clean accuracy is higher.
* **Larger $\epsilon$ than in training.** The AT model still has 19.4% at 12/255 and 14.6% at 16/255 (the standard model has 0.5% and 0.1%). The robustness decreases smoothly beyond the training $\epsilon$; it does not suddenly fail.

### Part (b): PGD vs FGSM, and different $\epsilon$

* **Standard model: PGD is much stronger than FGSM.** At 8/255, FGSM leaves 5.9% and PGD 2.4%. At 16/255, FGSM leaves 2.8% and PGD 0.1%.
  The loss surface of the standard model is curved near the data. One step in the direction of the gradient at $x$ does not find the maximum in the ball, and several smaller steps that follow the curvature do.
* **AT model: FGSM is almost as strong as PGD.** For $\epsilon \le 8/255$, the two attacks are within 0.4 points. At 4/255 and 8/255, PGD is even 0.3–0.4 points higher, which is noise from the random start and from the 20-step budget.
  Adversarial training makes the loss almost linear in the training ball, so the first gradient step already points to the worst case. Further steps add little.
  For $\epsilon$ larger than the training $\epsilon$ (12/255 and 16/255), PGD becomes stronger than FGSM again (19.4 vs 20.5, and 14.6 vs 16.6), because the loss is no longer linear so far from the training region.
  So the adversarially trained model is robust against both attacks. FGSM is a reliable estimate only inside the training $\epsilon$; PGD is needed to measure robustness for larger $\epsilon$.
* **Compared with the original model.** Against the standard model, PGD at 8/255 lowers the accuracy by 52.0 points (54.4 → 2.4). Against the AT model, it lowers the accuracy by 13.4 points (39.1 → 25.7). Against the standard model, FGSM at 8/255 lowers the accuracy by 48.5 points, and against the AT model by 13.8 points. Both attacks are much less effective after adversarial training.
* **Different training $\epsilon$** (rows 4/255 and 16/255):
  * There is a trade-off between clean accuracy and robustness. The clean accuracy goes down as the training $\epsilon$ goes up: 54.4 (standard) → 45.2 (4/255) → 39.1 (8/255) → 29.2 (16/255).
  * Each model is best near the $\epsilon$ it was trained for.
    * $\epsilon$ = 2/255 and 4/255: the 4/255 model is best (39.6% PGD at 2/255, 34.3% at 4/255). At 1/255, the standard model is still best (43.3%).
    * $\epsilon = 8/255$: the 8/255 model is best (25.7% PGD, against 23.8% and 24.5%).
    * Large $\epsilon$: the 16/255 model is best (21.7% PGD at 16/255, against 14.6% and 8.4%).
  * A larger training $\epsilon$ gives a flatter accuracy curve. From 0 to 16/255, the 16/255 model loses only 7.5 points, the 8/255 model 24.5 points, and the 4/255 model 36.8 points.
  * The 16/255 model is close to the limit of what 5 epochs can learn. Its training loss stays at 2.20 (random guessing is $\ln 10 \approx 2.30$), and its clean accuracy is low. With a training $\epsilon$ that is too large, short training mostly fails to fit the data.
  * For the 4/255 model, FGSM overestimates the robustness for $\epsilon$ larger than its training $\epsilon$ (16/255: 12.1% FGSM vs 8.4% PGD). For the 16/255 model, FGSM and PGD agree for every $\epsilon$ we tested. This is the same effect as above: FGSM ≈ PGD inside the training ball, and PGD is stronger outside it.